In [3]:
%%capture
!pip install -U dspy -Ue
!pip install -U python-dotenv
!pip install torch
!pip install transformers
!pip install accelerate
!pip install -q bitsandbytes trl peft accelerate
!pip install evaluate
!pip install transformers[sentencepiece]
!pip install scikit-learn

In [8]:
import dspy
lm = dspy.HFModel(model = "khalidrajan/Llama-3.1-8B-Instruct-Legal-NLI-Finetuned")

OSError: khalidrajan/Llama-3.1-8B-Instruct-Legal-NLI-Finetuned is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [ ]:
dspy.settings.configure(lm=lm, do_sample = False)

In [ ]:
import pandas as pd
df = pd.read_csv("hf://datasets/darrow-ai/LegalLensNLI-SharedTask/NLI.csv")

In [ ]:
df

In [ ]:
# Remove unnecessary columns
df = df[["premise", "hypothesis", "label"]]

In [ ]:
from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(df, test_size=0.3, random_state=42)

In [ ]:
train_df

In [ ]:
eval_df

In [ ]:
train_dataset = []
for premise, hypothesis, label in train_df.values:
    train_dataset.append(dspy.Example(premise=premise, hypothesis=hypothesis, label=label).with_inputs("premise", "hypothesis"))

In [ ]:
eval_dataset = []
for premise, hypothesis, label in eval_df.values:
    eval_dataset.append(dspy.Example(premise=premise, hypothesis=hypothesis, label=label).with_inputs("premise", "hypothesis"))

In [ ]:
from typing import Literal
class NLI(dspy.Signature):
    """
    Please classify the relationship between a legal premise and a hypothesis into one of three categories: Entailed, Contradict, Neutral.
    """
    premise: str = dspy.InputField()
    hypothesis: str = dspy.InputField()
    label: Literal["Entailed", "Contradict", "Neutral"] = dspy.OutputField()

In [ ]:
predictor = dspy.Predict(NLI)

In [ ]:
predictor(premise=train_dataset[0].premise, hypothesis=train_dataset[0].hypothesis)